# 02 Abnormal returns (RQ1)

Baseline abnormal returns of Chapter 5 and the CAAR tables. Inference:

- $SCAR_i = CAR_i / \left(\hat\sigma_i \sqrt{L + x_w'(X'X)^{-1}x_w}\right)$ (Equation 4.8), BMP $t$-test on the mean $SCAR_i$.
- KP correction for cross-sectional correlation, scaled by the number of ordered same-date event pairs
  $S=\sum_d n_d(n_d-1)$, so that $t_{KP} = t_{BMP}\sqrt{\dfrac{1-\bar\rho S/(N(N-1))}{1+\bar\rho S/N}}$.
  The correction applies only when $\bar\rho > 0$.
- $\bar\rho$ is the mean correlation of estimation-window residuals across same-date event pairs.
- Value-weighted tests use Kish effective N and are descriptive only.

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as st

from src.common import (TABLES, K_VALS, WINDOWS, ar_col, date_col, lev_col, window_cols, star, p_two_sided,
                        caar_stats, kp_factor, load_sample, ResidualCorrelation)

df = load_sample()
rc = ResidualCorrelation()
print(f"{len(df)} events")

615 events


## Groups

Leave-one-out rows: the joint NVIDIA-Intel announcement of 18 September 2025 creates two sample events
(INTC and NVDA), and the ex-INTC rows drop both. Nokia's conference appearance of 30 April 2026 carries most
of the European $[-1,+1]$ mean. `US ex-foreign` drops the 19 events of the five S&P 500 constituents
domiciled outside the US. The `MA ex-INTC` and `EU ex-Nokia` rows use the BMP test without the KP correction.

In [2]:
us, eu = df["Region"] == "US", df["Region"] == "EU"
tb = df["third_bridge"]
intc_pair = df["base"].isin(["INTC", "NVDA"]) & (df["Date"] == pd.Timestamp("2025-09-18"))
nokia = (df["base"] == "NOKIA") & (df["Date"] == pd.Timestamp("2026-04-30"))
foreign = df["base"].isin(["ACN", "NXPI", "STX", "MDT", "APTV"])
et = df["Event_Type"]

GROUPS = {
    "Full sample": df, "US": df[us], "EU": df[eu],
    "MA": df[et == "MA"], "CP": df[et == "CP"], "CM": df[et == "CM"], "CS": df[et == "CS"],
    "Full ex-TB": df[~tb], "US ex-TB": df[us & ~tb], "EU ex-TB": df[eu & ~tb], "Third Bridge only": df[tb],
    "US ex-foreign": df[us & ~foreign],
    "Firm-hosted": df[df["firm_hosted"]], "Firm-hosted ex-INTC": df[df["firm_hosted"] & ~intc_pair],
    "MA ex-INTC": df[(et == "MA") & ~intc_pair], "EU ex-Nokia": df[eu & ~nokia],
}
BMP_ONLY = {"MA ex-INTC", "EU ex-Nokia"}
RHO = {g: rc.rho_bar(sub) for g, sub in GROUPS.items() if g not in BMP_ONLY}
print(pd.Series(RHO).round(4).to_string())

Full sample            0.0302
US                     0.0350
EU                     0.1154
MA                     0.0792
CP                     0.0329
CM                    -0.0111
CS                     0.0887
Full ex-TB             0.0312
US ex-TB               0.0362
EU ex-TB               0.1098
Third Bridge only     -0.0071
US ex-foreign          0.0305
Firm-hosted            0.1470
Firm-hosted ex-INTC    0.1495


## CAAR by group and window

In [3]:
rows = []
for g, sub in GROUPS.items():
    for w, k1, k2 in WINDOWS:
        r = caar_stats(sub, k1, k2, rho=RHO.get(g))
        rows.append({"group": g, "window": w, **r, "sig_used": star(r.get("p_used"))})
caar = pd.DataFrame(rows)
caar.to_csv(TABLES / "caar.csv", index=False)
caar[caar["group"].isin(["Full sample", "US", "EU"])][
    ["group", "window", "N", "EW_CAAR", "t_used", "p_used", "VW_CAAR", "p_VW"]]

,group,window,N,EW_CAAR,t_used,p_used,VW_CAAR,p_VW
0,Full sample,"[-1,+1]",615,-0.0083,-0.6734,0.5010,-0.0452,0.8071
1,Full sample,"[0,+1]",615,-0.1487,-1.4484,0.1480,0.0246,0.8743
2,Full sample,"[0,+3]",615,-0.2693,-2.0931,0.0367,0.0166,0.9414
3,Full sample,"[-2,+2]",615,-0.0492,-1.1814,0.2379,-0.0964,0.6843
4,Full sample,"[-3,+3]",615,-0.2800,-2.4546,0.0144,-0.2520,0.3492
5,Full sample,"[-4,+4]",615,-0.4127,-2.8479,0.0045,-0.3377,0.2559
6,Full sample,"[-5,+5]",615,-0.2764,-2.2185,0.0269,-0.2344,0.4785
7,US,"[-1,+1]",545,-0.0666,-0.7620,0.4464,-0.0523,0.7794
8,US,"[0,+1]",545,-0.2079,-1.5812,0.1144,0.0225,0.8864
9,US,"[0,+3]",545,-0.3203,-2.0489,0.0409,0.0215,0.9255


## Benjamini-Hochberg across the seven windows, per region

In [4]:
bh = []
for g in ("Full sample", "US", "EU"):
    sub = caar[caar["group"] == g].dropna(subset=["p_used"]).sort_values("p_used").reset_index(drop=True)
    m = len(sub)
    p = sub["p_used"].to_numpy(float)
    q = np.minimum(np.minimum.accumulate((p * m / np.arange(1, m + 1))[::-1])[::-1], 1.0)
    bh += [dict(group=g, window=w, p_used=p_, q_BH=float(q_), sig_BH=star(float(q_)))
           for w, p_, q_ in zip(sub["window"], p, q)]
bh = pd.DataFrame(bh)
bh.to_csv(TABLES / "caar_window_bh.csv", index=False)
bh

,group,window,p_used,q_BH,sig_BH
0,Full sample,"[-4,+4]",0.0045,0.0318,**
1,Full sample,"[-3,+3]",0.0144,0.0503,*
2,Full sample,"[-5,+5]",0.0269,0.0627,*
3,Full sample,"[0,+3]",0.0367,0.0643,*
4,Full sample,"[0,+1]",0.1480,0.2072,
5,Full sample,"[-2,+2]",0.2379,0.2776,
6,Full sample,"[-1,+1]",0.5010,0.5010,
7,US,"[-4,+4]",0.0042,0.0216,**
8,US,"[-3,+3]",0.0062,0.0216,**
9,US,"[-5,+5]",0.0129,0.0301,**


## KP correction with overlapping windows (United States, $[-3,+3]$)

$S$ counts ordered pairs of events on the same calendar date. Counting every ordered pair whose $[-3,+3]$
windows share a trading day gives an upper bound on the dependence. The test is repeated with that $S$ (Section 4.2.3).

In [5]:
ks = window_cols(-3, 3)
z = GROUPS["US"].dropna(subset=ks + ["Sigma", lev_col(-3, 3)])
z = z[z["Sigma"] > 0]
n, rho = len(z), RHO["US"]
sar = z[ks].sum(axis=1) / (z["Sigma"] * 100 * np.sqrt(len(ks) + z[lev_col(-3, 3)]))
t_bmp = sar.mean() / (sar.std(ddof=1) / np.sqrt(n))
windows = [set(pd.to_datetime(w)) for w in z[[date_col(k) for k in range(-3, 4)]].to_numpy()]
s_overlap = sum(1 for i in range(n) for j in range(n) if i != j and windows[i] & windows[j])
s_same = kp_factor(z, rho, n)[1]
rows = []
for label, s in [("same-date pairs", s_same), ("overlapping windows", s_overlap)]:
    factor = np.sqrt((1 - rho * s / (n * (n - 1))) / (1 + rho * s / n))
    rows.append(dict(pairs=label, S=int(s), rho_bar=rho, t_KP=(t_bmp * factor),
                     p_KP=p_two_sided(t_bmp * factor, n - 1)))
kp_overlap = pd.DataFrame(rows)
kp_overlap.to_csv(TABLES / "kp_overlap_sensitivity.csv", index=False)
kp_overlap

,pairs,S,rho_bar,t_KP,p_KP
0,same-date pairs,2874,0.035,-2.7499,0.0062
1,overlapping windows,8572,0.035,-2.4031,0.0166


## Daily abnormal returns

Cross-sectional t-test on $AR_k$, and the KP-corrected test on $AR_k/\hat\sigma_i$.

In [6]:
rows = []
for g in ("Full sample", "US", "EU"):
    sub, rho = GROUPS[g], RHO[g]
    for k in K_VALS:
        v = sub[ar_col(k)].dropna()
        n = len(v)
        se = v.std(ddof=1) / np.sqrt(n)
        t = v.mean() / se
        p = p_two_sided(t, n - 1)
        row = dict(group=g, k=k, N=n, mean_AR=v.mean(), se=se,
                   t=t, p=p, sig=star(p))
        z = sub.dropna(subset=[ar_col(k), "Sigma"])
        z = z[z["Sigma"] > 0]
        sar = z[ar_col(k)] / (z["Sigma"] * 100)
        t_bmp = sar.mean() / (sar.std(ddof=1) / np.sqrt(len(z)))
        factor, pairs = kp_factor(z, rho, len(z))
        t_used = t_bmp * factor if (rho > 0 and np.isfinite(factor)) else t_bmp
        row.update(rho_bar=rho, same_date_pairs=int(pairs), t_BMP=t_bmp,
                   t_used=t_used, p_used=p_two_sided(t_used, len(z) - 1),
                   stat="KP" if rho > 0 and np.isfinite(factor) else "BMP")
        rows.append(row)
daily = pd.DataFrame(rows)
daily.to_csv(TABLES / "daily_ar.csv", index=False)
daily[daily["group"] == "US"]

,group,k,N,mean_AR,se,t,p,sig,rho_bar,same_date_pairs,t_BMP,t_used,p_used,stat
11,US,-5,545,-0.0231,0.0803,-0.2871,0.7742,,0.035,2874,-0.7878,-0.7238,0.4695,KP
12,US,-4,545,-0.0486,0.0738,-0.6589,0.5102,,0.035,2874,-1.0910,-1.0023,0.3166,KP
13,US,-3,545,-0.1753,0.0766,-2.2892,0.0225,**,0.035,2874,-2.8902,-2.6553,0.0082,KP
14,US,-2,545,-0.0653,0.0597,-1.0932,0.2748,,0.035,2874,-1.8714,-1.7193,0.0861,KP
15,US,-1,545,0.1413,0.0661,2.1389,0.0329,**,0.035,2874,1.3820,1.2697,0.2047,KP
16,US,0,545,-0.0520,0.0875,-0.5936,0.5530,,0.035,2874,-0.9751,-0.8959,0.3707,KP
17,US,1,545,-0.1559,0.0755,-2.0645,0.0394,**,0.035,2874,-1.5484,-1.4225,0.1554,KP
18,US,2,545,-0.0302,0.0670,-0.4508,0.6523,,0.035,2874,-0.4140,-0.3803,0.7039,KP
19,US,3,545,-0.0822,0.0675,-1.2180,0.2237,,0.035,2874,-1.5141,-1.3910,0.1648,KP
20,US,4,545,-0.0491,0.0792,-0.6202,0.5354,,0.035,2874,-1.0517,-0.9662,0.3344,KP


## Daily value-weighted and day-portfolio tests

Day portfolio: average the events on each calendar date first, then test across dates.

In [7]:
rows = []
for g in ("Full sample", "US", "EU"):
    sub = GROUPS[g]
    for k in K_VALS:
        z = sub.dropna(subset=[ar_col(k), "MktCap_USDm"])
        ar = z[ar_col(k)]
        w = z["MktCap_USDm"].to_numpy(float)
        w = w / w.sum()
        vw = float((ar.to_numpy() * w).sum())
        eff = 1.0 / np.sum(w ** 2)
        wvar = float((w * (ar.to_numpy() - vw) ** 2).sum()) * (eff / (eff - 1))
        t_vw = vw / np.sqrt(wvar / eff) if wvar > 0 else np.nan
        port = ar.groupby(z["Date"]).mean()
        t_dp = port.mean() / (port.std(ddof=1) / np.sqrt(len(port)))
        rows.append(dict(group=g, k=k, N=len(z), VW_mean_AR=vw, t_VW=t_vw,
                         p_VW=p_two_sided(t_vw, max(int(round(eff)) - 1, 1)), eff_N=eff,
                         N_dates=len(port), port_mean_AR=port.mean(),
                         t_port=t_dp, p_port=p_two_sided(t_dp, len(port) - 1)))
pd.DataFrame(rows).to_csv(TABLES / "daily_ar_vw_portfolio.csv", index=False)

rows = []
for g in ("Full sample", "US", "EU", "US ex-TB"):
    sub = GROUPS[g]
    for w, k1, k2 in WINDOWS:
        ks = window_cols(k1, k2)
        z = sub.dropna(subset=ks)
        car = z[ks].sum(axis=1)
        port = car.groupby(z["Date"]).mean()
        t_dp = port.mean() / (port.std(ddof=1) / np.sqrt(len(port)))
        rows.append(dict(group=g, window=w, N=len(z), EW_CAAR=float(car.mean()),
                         N_dates=len(port), port_mean_CAR=float(port.mean()),
                         t_port=float(t_dp), p_port=float(p_two_sided(t_dp, len(port) - 1)),
                         p_wilcoxon=float(st.wilcoxon(car).pvalue),
                         share_negative=float((car < 0).mean())))
day_port = pd.DataFrame(rows)
day_port.to_csv(TABLES / "day_portfolio_car.csv", index=False)
day_port[day_port["group"] == "US"]

,group,window,N,EW_CAAR,N_dates,port_mean_CAR,t_port,p_port,p_wilcoxon,share_negative
7,US,"[-1,+1]",545,-0.0666,334,-0.1520,-1.0110,0.3127,0.3948,0.5229
8,US,"[0,+1]",545,-0.2079,334,-0.2103,-1.6319,0.1036,0.0418,0.5431
9,US,"[0,+3]",545,-0.3203,334,-0.3270,-1.9477,0.0523,0.0876,0.5229
10,US,"[-2,+2]",545,-0.1621,334,-0.2693,-1.4511,0.1477,0.2191,0.5321
11,US,"[-3,+3]",545,-0.4196,334,-0.4816,-2.3652,0.0186,0.0091,0.5468
12,US,"[-4,+4]",545,-0.5174,334,-0.5967,-2.5857,0.0101,0.0498,0.5394
13,US,"[-5,+5]",545,-0.4383,334,-0.4642,-1.8110,0.0710,0.0660,0.5193


## Conference-day clustering

In [8]:
cp = GROUPS["CP"]
on = df["on_cluster_day"] == 1
print(f"CP events on the ten cluster days: {cp['on_cluster_day'].mean():.1%}")
rows = []
for g, sub in [("Full on-cluster", df[on]), ("Full off-cluster", df[~on]),
               ("CP on-cluster", cp[cp["on_cluster_day"] == 1]),
               ("CP off-cluster", cp[cp["on_cluster_day"] == 0])]:
    rho = rc.rho_bar(sub)
    for w, k1, k2 in [("[-1,+1]", -1, 1), ("[-5,+5]", -5, 5)]:
        rows.append({"group": g, "window": w, **caar_stats(sub, k1, k2, rho=rho)})
for w, k1, k2 in [("[-1,+1]", -1, 1), ("[-5,+5]", -5, 5)]:
    ks = window_cols(k1, k2)
    z = df.dropna(subset=ks)
    day_car = z[ks].sum(axis=1).groupby(z["Date"]).mean()
    n = len(day_car)
    t = day_car.mean() / (day_car.std(ddof=1) / np.sqrt(n))
    rows.append({"group": "Day portfolios (full)", "window": w, "N": n,
                 "EW_CAAR": day_car.mean(), "t_EW": t, "p_EW": p_two_sided(t, n - 1)})
clustering = pd.DataFrame(rows)
clustering.to_csv(TABLES / "clustering.csv", index=False)
clustering

CP events on the ten cluster days: 59.2%


,group,window,N,EW_CAAR,t_EW,p_EW,VW_CAAR,t_VW,p_VW,eff_N,rho_bar,same_date_pairs,t_used,p_used,stat
0,Full on-cluster,"[-1,+1]",159,0.2656,0.5573,0.5781,-0.3601,-0.8700,0.3920,27.9283,0.0301,3296.0,0.4365,0.6631,KP
1,Full on-cluster,"[-5,+5]",159,-0.2473,-1.2991,0.1958,-1.4641,-1.4931,0.1470,27.9283,0.0301,3296.0,-1.0175,0.3105,KP
2,Full off-cluster,"[-1,+1]",456,-0.1038,-1.1949,0.2327,0.0107,0.0521,0.9585,166.6896,0.0310,342.0,-1.1812,0.2381,KP
3,Full off-cluster,"[-5,+5]",456,-0.2866,-2.0278,0.0432,-0.0162,-0.0467,0.9628,166.6896,0.0310,342.0,-2.0046,0.0456,KP
4,CP on-cluster,"[-1,+1]",154,0.2587,0.4806,0.6315,-0.3657,-0.8713,0.3916,27.4102,0.0317,3100.0,0.3747,0.7084,KP
5,CP on-cluster,"[-5,+5]",154,-0.3186,-1.3684,0.1732,-1.4761,-1.4926,0.1476,27.4102,0.0317,3100.0,-1.0670,0.2876,KP
6,CP off-cluster,"[-1,+1]",106,-0.3329,-1.4769,0.1427,-0.5097,-1.1140,0.2755,26.7966,0.0518,196.0,-1.4102,0.1614,KP
7,CP off-cluster,"[-5,+5]",106,-0.7146,-2.1447,0.0343,0.7756,0.7457,0.4625,26.7966,0.0518,196.0,-2.0478,0.0431,KP
8,Day portfolios (full),"[-1,+1]",367,-0.1002,-0.6622,0.5083,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Day portfolios (full),"[-5,+5]",367,-0.3086,-1.2127,0.2260,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Reaction magnitude by tier

$VR_i = \sum_{k=-1}^{1} AR_{ik}^2 / (3\hat\sigma_i^2)$. Mann-Whitney tests compare each tier with the rest.

In [9]:
ar0 = df[ar_col(0)]
vr = ((df[window_cols(-1, 1)] / 100) ** 2).sum(axis=1) / (df["Sigma"] ** 2 * 3)


def tier_row(name, m, rest):
    a = ar0[m]
    row = dict(tier=name, N=int(m.sum()), mean_AR0=float(a.mean()),
               median_AR0=float(a.median()), mean_absAR0=float(a.abs().mean()),
               share_absAR0_gt2=float((a.abs() > 2).mean()),
               share_AR0_positive=float((a > 0).mean()),
               VR_mean=float(vr[m].mean()), VR_median=float(vr[m].median()))
    if name != "Firm-hosted ex-INTC":
        sar0 = a / (df.loc[m, "Sigma"] * 100)
        t0 = sar0.mean() / (sar0.std(ddof=1) / np.sqrt(len(sar0))) if len(sar0) > 2 else np.nan
        row.update(t_BMP_day0=float(t0) if np.isfinite(t0) else np.nan,
                   p_BMP_day0=float(p_two_sided(t0, len(sar0) - 1)) if np.isfinite(t0) else np.nan)
    row.update(mw_p_absAR0_vs_rest=float(st.mannwhitneyu(a.abs(), ar0[rest].abs()).pvalue),
               mw_p_VR_vs_rest=float(st.mannwhitneyu(vr[m], vr[rest]).pvalue))
    return row


rows = [tier_row(t, df["tier"] == t, df["tier"] != t)
        for t in ["Firm-hosted", "Conference appearance", "Research-provider or broker session",
                  "Third Bridge", "Other"]]
rows.append(tier_row("Firm-hosted ex-INTC", df["firm_hosted"] & ~intc_pair, ~df["firm_hosted"]))
tiers = pd.DataFrame(rows)
tiers.to_csv(TABLES / "tier_magnitude.csv", index=False)
tiers

,tier,N,mean_AR0,median_AR0,mean_absAR0,share_absAR0_gt2,share_AR0_positive,VR_mean,VR_median,t_BMP_day0,p_BMP_day0,mw_p_absAR0_vs_rest,mw_p_VR_vs_rest
0,Firm-hosted,42,0.4764,0.3323,2.2072,0.4048,0.5714,1.8015,0.8922,0.8110,0.4221,0.0047,0.0162
1,Conference appearance,248,-0.0700,-0.0934,1.4111,0.2339,0.4839,1.3125,0.7061,-1.2904,0.1981,0.0050,0.0864
2,Research-provider or broker session,125,-0.1995,-0.1303,1.0641,0.1280,0.4560,1.1130,0.5969,-1.4426,0.1517,0.0865,0.5490
3,Third Bridge,198,0.0085,-0.0307,1.0524,0.1364,0.4848,1.1009,0.4995,0.0820,0.9347,0.0017,0.0094
4,Other,2,-4.6307,-4.6307,4.6307,0.5000,0.0000,2.9378,2.9378,NaN,NaN,0.2469,0.9001
5,Firm-hosted ex-INTC,40,-0.1189,0.2037,1.6984,0.3750,0.5500,1.4206,0.8506,NaN,NaN,0.0169,0.0482


## Firm concentration and firm-level tests

Firm-level test: one observation per firm, the firm's mean CAR. Firm-clustered test: the event-level CAAR
with standard errors clustered by firm.

In [10]:
label = df.drop_duplicates("firm_key").set_index("firm_key")["base"].to_dict()
conc = df.groupby("firm_key").size().sort_values(ascending=False)
pd.DataFrame([dict(rank=i + 1, firm=label[f], firm_key=f, n_events=int(v),
                   share_pct=(100 * v / len(df))) for i, (f, v) in enumerate(conc.items())]
             ).to_csv(TABLES / "firm_concentration.csv", index=False)
print(f"{df['firm_key'].nunique()} firms (US {df[us]['firm_key'].nunique()}, EU {df[eu]['firm_key'].nunique()}); "
      f"top 5 hold {conc.head(5).sum()} events; repeat announcers {(conc > 1).sum()} with {conc[conc > 1].sum()} events")


def cluster_se(x, groups):
    x = np.asarray(x, float)
    s = pd.Series(x - x.mean()).groupby(np.asarray(groups)).sum()
    g = len(s)
    return np.sqrt(float((s ** 2).sum()) * (g / (g - 1)) / len(x) ** 2), g


rows = []
for g in ("Full sample", "US", "EU"):
    for w, k1, k2 in [("[-1,+1]", -1, 1), ("[-3,+3]", -3, 3), ("[-5,+5]", -5, 5)]:
        ks = window_cols(k1, k2)
        z = GROUPS[g].dropna(subset=ks)
        car = z[ks].sum(axis=1)
        fm = car.groupby(z["firm_key"]).mean()
        t_f = fm.mean() / (fm.std(ddof=1) / np.sqrt(len(fm)))
        se_c, g_c = cluster_se(car, z["firm_key"])
        t_c = car.mean() / se_c
        rows.append(dict(group=g, window=w, N_events=len(z), N_firms=len(fm),
                         firm_mean_CAAR=fm.mean(), t_firm=t_f,
                         p_firm=p_two_sided(t_f, len(fm) - 1), event_CAAR=car.mean(),
                         t_clustered=t_c, p_clustered=p_two_sided(t_c, g_c - 1)))
firm_level = pd.DataFrame(rows)
firm_level.to_csv(TABLES / "firm_level_caar.csv", index=False)
firm_level

174 firms (US 133, EU 41); top 5 hold 212 events; repeat announcers 93 with 534 events


,group,window,N_events,N_firms,firm_mean_CAAR,t_firm,p_firm,event_CAAR,t_clustered,p_clustered
0,Full sample,"[-1,+1]",615,174,0.3256,1.7510,0.0817,-0.0083,-0.0639,0.9491
1,Full sample,"[-3,+3]",615,174,0.0100,0.0351,0.9720,-0.2800,-1.3355,0.1835
2,Full sample,"[-5,+5]",615,174,-0.0159,-0.0426,0.9660,-0.2764,-0.9377,0.3497
3,US,"[-1,+1]",545,133,0.3199,1.5133,0.1326,-0.0666,-0.5122,0.6094
4,US,"[-3,+3]",545,133,-0.1596,-0.4851,0.6284,-0.4196,-1.8890,0.0611
5,US,"[-5,+5]",545,133,-0.2564,-0.5949,0.5530,-0.4383,-1.3776,0.1707
6,EU,"[-1,+1]",70,41,0.3440,0.8704,0.3893,0.4455,0.9573,0.3442
7,EU,"[-3,+3]",70,41,0.5604,0.9793,0.3333,0.8071,1.4243,0.1621
8,EU,"[-5,+5]",70,41,0.7641,1.0309,0.3088,0.9837,1.5030,0.1407
